In [1]:
# Setting up the file path
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

In [2]:
# Import packages and modules
import numpy as np
import torch
from RL4CRN.iocrns.mass_action_iocrn import MassActionIOCRN
from RL4CRN.policies.add_reaction_from_library import AddReactionFromLibrary

In [3]:
# Construct the basic CRN
species_labels = ['X_1', 'Z_1', 'Z_2']
inputs_labels = ['u_1', 'u_2', 'u_3', 'u_4']
S_R = np.array([[0], [1], [0]], dtype=np.int8)
S_P = np.array([[1], [1], [0]], dtype=np.int8)
k = 1
c = np.array([k], dtype=np.float32)
S_I = np.array([[0], [0], [0], [0]], dtype=np.int8)
o = np.array([1], dtype=np.int8)
iocrn_0 = MassActionIOCRN(S_R, S_P, c, S_I, o, species_labels, inputs_labels)

# Add reactions to CRN_1
iocrn_1 = iocrn_0.clone()
iocrn_1.add_reaction({'reactant1 index': 0, 'reactant2 index': 1, 'product1 index': 0, 'product2 index': 0, 'input influence index': 0, 'rate constant':0.1}, mode='species index')
iocrn_1.add_reaction({'reactant1 index': 1, 'reactant2 index': 1, 'product1 index': 2, 'product2 index': 3, 'input influence index': 3, 'rate constant':0.5}, mode='species index')
iocrn_1.add_reaction({'reactant1 index': 1, 'reactant2 index': 3, 'product1 index': 0, 'product2 index': 3, 'input influence index': 2, 'rate constant':0.7}, mode='species index')

# Add reactions to CRN_2
iocrn_2 = iocrn_0.clone()
iocrn_2.add_reaction({'reactant1 index': 2, 'reactant2 index': 3, 'product1 index': 0, 'product2 index': 1, 'input influence index': 2, 'rate constant':0.1}, mode='species index')
iocrn_2.add_reaction({'reactant1 index': 2, 'reactant2 index': 3, 'product1 index': 0, 'product2 index': 3, 'input influence index': 4, 'rate constant':0.1}, mode='species index')
iocrn_2.add_reaction({'reactant1 index': 0, 'reactant2 index': 1, 'product1 index': 2, 'product2 index': 2, 'input influence index': 0, 'rate constant':0.3}, mode='species index')

# Add reactions to CRN_3
iocrn_3 = iocrn_0.clone()
iocrn_3.add_reaction({'reactant1 index': 0, 'reactant2 index': 0, 'product1 index': 0, 'product2 index': 3, 'input influence index': 0, 'rate constant':0.8}, mode='species index')
iocrn_3.add_reaction({'reactant1 index': 3, 'reactant2 index': 3, 'product1 index': 1, 'product2 index': 3, 'input influence index': 3, 'rate constant':0.2}, mode='species index')
iocrn_3.add_reaction({'reactant1 index': 0, 'reactant2 index': 2, 'product1 index': 1, 'product2 index': 1, 'input influence index': 3, 'rate constant':0.9}, mode='species index')

# Create list of CRNs to represent a batch
iocrn_list = [iocrn_1, iocrn_2, iocrn_3]

In [4]:
# Construct the Policy Model
n = 3; p = 4
encoder_attributes = {"hidden_size": 64, "num_layers": 2}
structure_decoder_attributes = {"hidden_size": 64, "num_layers": 2}
rate_decoder_attributes = {"hidden_size": 64, "num_layers": 2}
input_influence_decoder_attributes = {"hidden_size": 64, "num_layers": 2}
deep_layer_size = 1024
M = iocrn_0.get_reactions_range()
policy = AddReactionFromLibrary(M, p, encoder_attributes, deep_layer_size, structure_decoder_attributes, rate_decoder_attributes, input_influence_decoder_attributes, continuous_distribution='lognormal', allow_input_influence=True)

In [5]:
# Collect observations from the batch of CRNs
N = len(iocrn_list)
reactions_indices_batch = np.array([crn.reactions_indices for crn in iocrn_list])
rate_constants_batch = np.array([crn.c for crn in iocrn_list])

def get_influenced_reactions_batch(iocrn_list, p):
    """
    Collects the influenced reactions for each input across a batch of IOCRNs.
    Args:
        iocrn_list (list): List of IOCRN_MassAction objects.
        num_inputs (int): Number of inputs in the IOCRNs.
    Returns:
        list: A list of p numpy arrays, each containing the influenced reactions for a specific input. 
        Each numpy array is associated with a specific input and has shape (N, #), where N is the batch size and # is the maximum number of reactions in any CRN in the batch influenced by this input.
    """
    influenced_reactions = []
    for i in range(p):
        rows = [np.array(crn.list_influenced_reactions[i]) for crn in iocrn_list]
        max_len = max((len(r) for r in rows), default=0)
        padded = [np.pad(r, (0, max_len - len(r)), constant_values=0) for r in rows]
        influenced_reactions.append(np.array(padded).astype(np.int64))
    return influenced_reactions

reactions_indices_influenced_by_inputs_batch = get_influenced_reactions_batch(iocrn_list, p)

observation_batch = reactions_indices_batch, rate_constants_batch, reactions_indices_influenced_by_inputs_batch

In [ ]:
# Run the Policy Model
samples, log_probability, entropy = policy(observation_batch, mode='full')

# Print the output
for i in range(N):
    print(f"Sample {i}:")
    print("Generated reaction structure:", samples[i]['reaction index'])
    print("Generated reaction rates:", samples[i]['rate constant'])
    print("Generated input influence:", samples[i]['input influence index'])
    print("log probability:", log_probability)
    print("Entropy:", entropy)
    print("---------------------")

In [ ]:
# Add samples to the CRNs
for i in range(N):
    # reaction is a dictionary with keys 'reaction index', 'input influence index', 'rate constant'
    reaction = samples[i]
    print('Before adding the generated reactions:')
    print(iocrn_list[i])
    iocrn_list[i].add_reaction(reaction, mode='reaction index')
    print('After adding the generated reactions:')
    print(iocrn_list[i])
    print('----------------------')
    print('---------------------------------------------------')